# FashionMNIST Two-Branch ANN on Colab T4

Notebook này tự chạy độc lập trên Colab. Nếu mount được Google Drive thì artifact sẽ được lưu vào Drive; nếu không thì lưu tạm trong `/content`.


In [8]:
import os
from pathlib import Path

IN_COLAB = 'COLAB_GPU' in os.environ
save_root = Path('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    preferred_roots = [
        Path('/content/drive/MyDrive/Colab Experiments/Lab3-LT'),
        Path('/content/drive/MyDrive/Lab3-LT'),
        Path('/content/drive/MyDrive/Colab Notebooks/Lab3-LT'),
    ]
    for candidate in preferred_roots:
        candidate.mkdir(parents=True, exist_ok=True)
        save_root = candidate
        break

artifact_dir = save_root / 'artifacts' / 'fashion_mnist_ann_two_branch'
cache_dir = save_root / 'data' / 'fashion_mnist_ann_two_branch_cache'
artifact_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

print('save_root =', save_root)
print('artifact_dir =', artifact_dir)
print('cache_dir =', cache_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
save_root = /content/drive/MyDrive/Colab Experiments/Lab3-LT
artifact_dir = /content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_two_branch
cache_dir = /content/drive/MyDrive/Colab Experiments/Lab3-LT/data/fashion_mnist_ann_two_branch_cache


In [9]:
if IN_COLAB:
    !pip install -q scikit-image opencv-python-headless
    !nvidia-smi


Wed Mar 25 19:21:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             30W /   70W |     231MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import json
import math
import random

import cv2
import numpy as np
import torch
import torch.nn as nn
from skimage.feature import hog
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import FashionMNIST


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_slant_angle(image: np.ndarray, threshold: float = 0.1) -> float:
    mask = image > threshold
    ys, xs = np.nonzero(mask)
    if len(xs) < 5:
        return 0.0
    weights = image[ys, xs].astype(np.float64)
    if weights.sum() <= 1e-8:
        return 0.0
    x_mean = np.average(xs, weights=weights)
    y_mean = np.average(ys, weights=weights)
    x_centered = xs - x_mean
    y_centered = ys - y_mean
    var_y = np.average(y_centered * y_centered, weights=weights)
    if var_y <= 1e-8:
        return 0.0
    cov_xy = np.average(x_centered * y_centered, weights=weights)
    return math.atan(cov_xy / var_y)


def deslant_image(image: np.ndarray) -> np.ndarray:
    angle = compute_slant_angle(image)
    shear = -math.tan(angle)
    if abs(shear) < 1e-4:
        return image.copy()
    h, w = image.shape
    center_y = (h - 1) / 2.0
    tx = -shear * center_y
    matrix = np.array([[1.0, shear, tx], [0.0, 1.0, 0.0]], dtype=np.float32)
    warped = cv2.warpAffine(image, matrix, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)


def recenter_image(image: np.ndarray, threshold: float = 0.1) -> np.ndarray:
    mask = image > threshold
    ys, xs = np.nonzero(mask)
    if len(xs) < 5:
        return image.copy()
    weights = image[ys, xs].astype(np.float64)
    if weights.sum() <= 1e-8:
        return image.copy()
    x_mean = np.average(xs, weights=weights)
    y_mean = np.average(ys, weights=weights)
    target_x = (image.shape[1] - 1) / 2.0
    target_y = (image.shape[0] - 1) / 2.0
    shift_x = target_x - x_mean
    shift_y = target_y - y_mean
    matrix = np.array([[1.0, 0.0, shift_x], [0.0, 1.0, shift_y]], dtype=np.float32)
    warped = cv2.warpAffine(image, matrix, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)


def random_affine(image: np.ndarray, max_rotate: float, max_shift: float, max_shear: float) -> np.ndarray:
    angle = random.uniform(-max_rotate, max_rotate)
    shear_deg = random.uniform(-max_shear, max_shear)
    tx = random.uniform(-max_shift, max_shift) * image.shape[1]
    ty = random.uniform(-max_shift, max_shift) * image.shape[0]
    center = ((image.shape[1] - 1) / 2.0, (image.shape[0] - 1) / 2.0)
    rot = cv2.getRotationMatrix2D(center, angle, 1.0)
    rot[0, 2] += tx
    rot[1, 2] += ty
    warped = cv2.warpAffine(image, rot, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    shear = math.tan(math.radians(shear_deg))
    center_y = (image.shape[0] - 1) / 2.0
    tx_shear = -shear * center_y
    shear_matrix = np.array([[1.0, shear, tx_shear], [0.0, 1.0, 0.0]], dtype=np.float32)
    warped = cv2.warpAffine(warped, shear_matrix, (image.shape[1], image.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return np.clip(warped, 0.0, 1.0)


def extract_hog_features(image: np.ndarray) -> np.ndarray:
    features = hog(image, orientations=9, pixels_per_cell=(4, 4), cells_per_block=(2, 2), block_norm='L2-Hys', transform_sqrt=False, feature_vector=True)
    return features.astype(np.float32)


def preprocess_image(image: np.ndarray, use_augmentation: bool) -> tuple[np.ndarray, np.ndarray]:
    proc = image.astype(np.float32) / 255.0
    if use_augmentation:
        proc = random_affine(proc, max_rotate=10.0, max_shift=0.08, max_shear=8.0)
    proc = deslant_image(proc)
    proc = recenter_image(proc)
    raw = proc.reshape(-1).astype(np.float32)
    hog_feat = extract_hog_features(proc)
    return raw, hog_feat


def load_or_create_cache(split_name: str, images: np.ndarray, labels: np.ndarray, augment: bool):
    suffix = 'aug' if augment else 'plain'
    cache_path = cache_dir / f'{split_name}_{suffix}.npz'
    if cache_path.exists():
        data = np.load(cache_path)
        return {key: data[key] for key in data.files}
    raw_list = []
    hog_list = []
    for image in images:
        raw, hog_feat = preprocess_image(image, use_augmentation=augment)
        raw_list.append(raw)
        hog_list.append(hog_feat)
    cache = {
        'raw': np.stack(raw_list).astype(np.float32),
        'hog': np.stack(hog_list).astype(np.float32),
        'labels': labels.astype(np.int64),
    }
    np.savez_compressed(cache_path, **cache)
    return cache


class FeatureDataset(Dataset):
    def __init__(self, raw_features, hog_features, labels, raw_mean, raw_std, hog_mean, hog_std):
        self.raw = (raw_features - raw_mean) / raw_std
        self.hog = (hog_features - hog_mean) / hog_std
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        raw = torch.from_numpy(self.raw[idx]).float()
        hog_feat = torch.from_numpy(self.hog[idx]).float()
        label = torch.tensor(self.labels[idx]).long()
        return raw, hog_feat, label


class Branch(nn.Module):
    def __init__(self, in_dim: int, hidden_dims: list[int], dropout: float) -> None:
        super().__init__()
        layers = []
        current_dim = in_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            ])
            current_dim = hidden_dim
        self.net = nn.Sequential(*layers)
        self.out_dim = current_dim
    def forward(self, x):
        return self.net(x)


class TwoBranchAnn(nn.Module):
    def __init__(self, raw_dim: int, hog_dim: int, dropout: float = 0.25) -> None:
        super().__init__()
        self.raw_branch = Branch(raw_dim, [512, 256], dropout)
        self.hog_branch = Branch(hog_dim, [768, 256], dropout)
        self.head = nn.Sequential(
            nn.Linear(self.raw_branch.out_dim + self.hog_branch.out_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 10),
        )
    def forward(self, raw_x, hog_x):
        raw_feat = self.raw_branch(raw_x)
        hog_feat = self.hog_branch(hog_x)
        fused = torch.cat([raw_feat, hog_feat], dim=1)
        return self.head(fused)


def run_epoch(model, loader, criterion, device, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for raw_x, hog_x, labels in loader:
        raw_x = raw_x.to(device)
        hog_x = hog_x.to(device)
        labels = labels.to(device)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        logits = model(raw_x, hog_x)
        loss = criterion(logits, labels)
        if optimizer is not None:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / max(total_samples, 1), total_correct / max(total_samples, 1)


def evaluate_test(model, loader, device):
    model.eval()
    total_correct, total_samples = 0, 0
    confusion = np.zeros((10, 10), dtype=np.int64)
    with torch.no_grad():
        for raw_x, hog_x, labels in loader:
            raw_x = raw_x.to(device)
            hog_x = hog_x.to(device)
            labels = labels.to(device)
            logits = model(raw_x, hog_x)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            for truth, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                confusion[truth, pred] += 1
    per_class_accuracy = {}
    for class_idx in range(10):
        total_in_class = confusion[class_idx].sum()
        acc = float(confusion[class_idx, class_idx] / total_in_class) if total_in_class > 0 else 0.0
        per_class_accuracy[str(class_idx)] = acc
    return total_correct / max(total_samples, 1), confusion, per_class_accuracy


set_seed(42)
data_root = save_root / 'data'
train_ds_raw = FashionMNIST(root=str(data_root), train=True, download=True)
test_ds_raw = FashionMNIST(root=str(data_root), train=False, download=True)

train_cache = load_or_create_cache('train', train_ds_raw.data.numpy(), train_ds_raw.targets.numpy(), augment=True)
test_cache = load_or_create_cache('test', test_ds_raw.data.numpy(), test_ds_raw.targets.numpy(), augment=False)

raw_mean = train_cache['raw'].mean(axis=0, dtype=np.float64).astype(np.float32)
raw_std = train_cache['raw'].std(axis=0, dtype=np.float64).astype(np.float32)
hog_mean = train_cache['hog'].mean(axis=0, dtype=np.float64).astype(np.float32)
hog_std = train_cache['hog'].std(axis=0, dtype=np.float64).astype(np.float32)
raw_std = np.where(raw_std < 1e-6, 1.0, raw_std)
hog_std = np.where(hog_std < 1e-6, 1.0, hog_std)

train_ds = FeatureDataset(train_cache['raw'], train_cache['hog'], train_cache['labels'], raw_mean, raw_std, hog_mean, hog_std)
test_ds = FeatureDataset(test_cache['raw'], test_cache['hog'], test_cache['labels'], raw_mean, raw_std, hog_mean, hog_std)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=device.type == 'cuda')
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=device.type == 'cuda')

model = TwoBranchAnn(raw_dim=train_cache['raw'].shape[1], hog_dim=train_cache['hog'].shape[1], dropout=0.25).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

history = []
for epoch in range(1, 51):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer=optimizer)
    scheduler.step()
    row = {'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc}
    history.append(row)
    print(json.dumps(row, ensure_ascii=False))

final_train_acc = history[-1]['train_acc']
final_train_loss = history[-1]['train_loss']
test_acc, confusion, per_class_accuracy = evaluate_test(model, test_loader, device)
generalization_gap = final_train_acc - test_acc

model_path = artifact_dir / 'best_model.pt'
metrics_path = artifact_dir / 'metrics.json'
confusion_path = artifact_dir / 'confusion_matrix.npy'
class_accuracy_path = artifact_dir / 'class_accuracy.json'

torch.save({
    'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
    'raw_mean': raw_mean,
    'raw_std': raw_std,
    'hog_mean': hog_mean,
    'hog_std': hog_std,
    'final_train_acc': final_train_acc,
    'final_train_loss': final_train_loss,
    'test_acc': test_acc,
    'generalization_gap': generalization_gap,
    'per_class_accuracy': per_class_accuracy,
}, model_path)

metrics = {
    'final_train_acc': final_train_acc,
    'final_train_loss': final_train_loss,
    'test_acc': test_acc,
    'generalization_gap': generalization_gap,
    'per_class_accuracy': per_class_accuracy,
    'epochs_ran': len(history),
    'history': history,
}
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
class_accuracy_path.write_text(json.dumps(per_class_accuracy, indent=2, ensure_ascii=False), encoding='utf-8')
np.save(confusion_path, confusion)
print('saved_to =', artifact_dir)


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 188kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.84MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.2MB/s]


device = cuda
{"epoch": 1, "train_loss": 0.6651252765655518, "train_acc": 0.8485833333333334}
{"epoch": 2, "train_loss": 0.5565746750831604, "train_acc": 0.8902666666666667}
{"epoch": 3, "train_loss": 0.5204246457417806, "train_acc": 0.9056666666666666}
{"epoch": 4, "train_loss": 0.4896551498730977, "train_acc": 0.9186666666666666}
{"epoch": 5, "train_loss": 0.46483591051101686, "train_acc": 0.93145}
{"epoch": 6, "train_loss": 0.4410854902903239, "train_acc": 0.9414833333333333}
{"epoch": 7, "train_loss": 0.42006908071835836, "train_acc": 0.9510833333333333}
{"epoch": 8, "train_loss": 0.40297390613555906, "train_acc": 0.9587333333333333}
{"epoch": 9, "train_loss": 0.3855837127049764, "train_acc": 0.9666333333333333}
{"epoch": 10, "train_loss": 0.3683267895380656, "train_acc": 0.9735833333333334}
{"epoch": 11, "train_loss": 0.35910068747202556, "train_acc": 0.97795}
{"epoch": 12, "train_loss": 0.34868298228581746, "train_acc": 0.9827}
{"epoch": 13, "train_loss": 0.34346699997584024, "tr

In [11]:
import json

metrics_path = artifact_dir / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('final_train_acc =', metrics['final_train_acc'])
print('test_acc =', metrics['test_acc'])
print('generalization_gap =', metrics['generalization_gap'])


final_train_acc = 0.9998666666666667
test_acc = 0.8969
generalization_gap = 0.10296666666666665


In [12]:
!ls -lah "$artifact_dir"
!find "$artifact_dir" -maxdepth 1 -type f


total 7.2M
-rw------- 1 root root 7.2M Mar 25 19:25 best_model.pt
-rw------- 1 root root  142 Mar 25 19:25 class_accuracy.json
-rw------- 1 root root  928 Mar 25 19:25 confusion_matrix.npy
-rw------- 1 root root 5.6K Mar 25 19:25 metrics.json
/content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_two_branch/best_model.pt
/content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_two_branch/metrics.json
/content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_two_branch/class_accuracy.json
/content/drive/MyDrive/Colab Experiments/Lab3-LT/artifacts/fashion_mnist_ann_two_branch/confusion_matrix.npy
